# Download files

In [ ]:
# Download sample text
# Download github repo of raptor

In [ ]:
git clone https://github.com/parthsarthi03/raptor

In [ ]:
import gdown

"https://drive.google.com/file/d/1_sE53mgV_RptRv8LXl4IPrAeruTfmw3-/view?usp=drive_link"

# Imports

In [1]:
with open('/home/oh/arubinstein17/.config/hugging_face/hf.yaml', 'r') as file:
    token = file.read()
    token = token.split(":")[1].strip()

In [2]:
import os
PATH_TO_RAPTOR = os.path.join(
    os.path.dirname(os.path.abspath(''))
)
HF_TOKEN = token
RANDOM_SEED = 42
TREE_PATH = "tree.pkl"
SAMPLE_TEXT_PATH = os.path.join(os.path.dirname(os.path.abspath('')), 'demo', 'sample.txt')
GRANULARITY = 100
SUMMARY_LENGTH = 200

In [3]:
%reload_ext autoreload
%autoreload 2


import os
import sys
# import argparse
import torch
from transformers import AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
import torch
from huggingface_hub import login
import numpy as np
import random
import logging
import pickle
import tiktoken
from typing import (
    Any,
    Callable,
    Dict,
    List,
    Optional,
    Set,
    Tuple
)
# import time
# import tqdm
import copy


sys.path.insert(
    0,
    PATH_TO_RAPTOR
)
# print(sys.path)
# from raptor import RetrievalAugmentation
from raptor import (
    BaseSummarizationModel,
    BaseQAModel,
    BaseEmbeddingModel,
    RetrievalAugmentationConfig,
    # RetrievalAugmentation
)
# from raptor.tree_structures import (
#     Tree
# )
from raptor.Retrievers import (
    BaseRetriever
)
from raptor.utils import (
    reverse_mapping,
    get_node_list,
    # get_embeddings,
    distances_from_embeddings,
    indices_of_nearest_neighbors_from_distances,
    get_text,
    split_text
)
# from raptor.tree_builder import (
#     TreeBuilder,
# )
# from raptor.tree_retriever import (
#     TreeRetrieverConfig
# )
from raptor.cluster_tree_builder import (
    ClusterTreeConfig
)
sys.path.pop(0)

/home/oh/arubinstein17/github/raptor/envs/raptor/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-02-27 01:44:04,945 - Loading faiss with AVX2 support.
2025-02-27 01:44:04,967 - Successfully loaded faiss with AVX2 support.
2025-02-27 01:44:04,977 - Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes.


'/mnt/lustre/work/oh/arubinstein17/github/raptor'

# Infrastructure

In [9]:
# You can define your own Summarization model by extending the base Summarization Class.
class GEMMASummarizationModel(BaseSummarizationModel):
    def __init__(self, model_name="google/gemma-2b-it"):
        # Initialize the tokenizer and the pipeline for the GEMMA model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.summarization_pipeline = pipeline(
            "text-generation",
            model=model_name,
            model_kwargs={"torch_dtype": torch.bfloat16},
            device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),  # Use "cpu" if CUDA is not available
        )

    def summarize(self, context, max_tokens=150):
        # Format the prompt for summarization
        messages=[
            {"role": "user", "content": f"Write a summary of the following, including as many key details as possible: {context}:"}
        ]

        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Generate the summary using the pipeline
        apply_random_seed(RANDOM_SEED)
        outputs = self.summarization_pipeline(
            prompt,
            max_new_tokens=max_tokens,
            # do_sample=True,
            do_sample=False, # tmp
            temperature=0.7,
            # temperature=0.0, # tmp
            top_k=50,
            top_p=0.95
        )

        # Extracting and returning the generated summary
        summary = outputs[0]["generated_text"].strip()
        # remove technical prefix
        split = summary.split("start_of_turn>model\n")
        summarization_prompt = "\n\n".join(split[:-1])
        summary = split[-1].strip()
        return summary, summarization_prompt


class GEMMAQAModel(BaseQAModel):
    def __init__(self, model_name= "google/gemma-2b-it"):
        # Initialize the tokenizer and the pipeline for the model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.qa_pipeline = pipeline(
            "text-generation",
            model=model_name,
            model_kwargs={"torch_dtype": torch.bfloat16},
            device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        )

    def answer_question(self, context, question):
        # Apply the chat template for the context and question
        messages=[
              {"role": "user", "content": f"Given Context: {context} Give the best full answer amongst the option to question {question}"}
        ]
        print("Context: ", context)
        print("Question: ", question)
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Generate the answer using the pipeline
        outputs = self.qa_pipeline(
            prompt,
            max_new_tokens=256,
            # do_sample=True,
            do_sample=False, # tmp
            temperature=0.7,
            # temperature=0.0, # tmp
            top_k=50,
            top_p=0.95
        )

        # Extracting and returning the generated answer
        answer = outputs[0]["generated_text"][len(prompt):]
        return answer


class SBertEmbeddingModel(BaseEmbeddingModel):
    def __init__(self, model_name="sentence-transformers/multi-qa-mpnet-base-cos-v1"):
        self.model = SentenceTransformer(model_name)

    def create_embedding(self, text):
        return self.model.encode(text)

    def __call__(self, text):
        return self.create_embedding(text)


def apply_random_seed(random_seed):
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # to suppress warning
    torch.use_deterministic_algorithms(True, warn_only=True)


class Node:
    """
    Represents a node in the hierarchical tree structure.
    """

    def __init__(self, text: str, index: int, children: Set[int], embeddings, summarization_prompt: Optional[str] = None) -> None:
        self.text = text
        self.index = index
        self.children = children
        self.embeddings = embeddings
        self.summarization_prompt = summarization_prompt

class Tree:
    """
    Represents the entire hierarchical tree structure.
    """

    def __init__(
        self, all_nodes, root_nodes, leaf_nodes, num_layers, layer_to_nodes
    ) -> None:
        self.all_nodes = all_nodes
        self.root_nodes = root_nodes
        self.leaf_nodes = leaf_nodes
        self.num_layers = num_layers
        self.layer_to_nodes = layer_to_nodes


class RetrievalAugmentation:
    """
    A Retrieval Augmentation class that combines the TreeBuilder and TreeRetriever classes.
    Enables adding documents to the tree, retrieving information, and answering questions.
    """

    def __init__(
        self,
        config=None,
        tree=None,
        embedding_models=None,
        cluster_embedding_model=None,
        retrieve_context_func=None
    ):
        """
        Initializes a RetrievalAugmentation instance with the specified configuration.
        Args:
            config (RetrievalAugmentationConfig): The configuration for the RetrievalAugmentation instance.
            tree: The tree instance or the path to a pickled tree file.
            retrieve_context_func: The function to retrieve the context for the question.
        """
        if config is None:
            config = RetrievalAugmentationConfig(
                tb_embedding_models=embedding_models,
                tb_cluster_embedding_model=cluster_embedding_model
            )
        if not isinstance(config, RetrievalAugmentationConfig):
            raise ValueError(
                "config must be an instance of RetrievalAugmentationConfig"
            )

        # Check if tree is a string (indicating a path to a pickled tree)
        if isinstance(tree, str) and os.path.exists(tree):
            try:
                with open(tree, "rb") as file:
                    self.tree = pickle.load(file)
                if not isinstance(self.tree, Tree):
                    raise ValueError("The loaded object is not an instance of Tree")
            except Exception as e:
                raise ValueError(f"Failed to load tree from {tree}: {e}")
        elif isinstance(tree, str) and not os.path.exists(tree):
            self.tree = None
        elif isinstance(tree, Tree) or tree is None:
            self.tree = tree
        else:
            raise ValueError(
                "tree must be an instance of Tree, a path to a pickled Tree, or None"
            )

        # tree_builder_class = supported_tree_builders[config.tree_builder_type][0]
        # tree_builder_class = ClusterTreeBuilder
        tree_builder_class = TreeBuilder
        self.tree_builder = tree_builder_class(config.tree_builder_config)

        self.tree_retriever_config = config.tree_retriever_config
        self.qa_model = config.qa_model

        self.retrieve_context_func = retrieve_context_func

        if self.tree is not None:
            self.retriever = TreeRetriever(
                self.tree_retriever_config,
                self.tree,
                retrieve_context_func=self.retrieve_context_func
            )
        else:
            self.retriever = None

        logging.info(
            f"Successfully initialized RetrievalAugmentation with Config {config.log_config()}"
        )

    def add_documents(self, docs):
        """
        Adds documents to the tree and creates a TreeRetriever instance.

        Args:
            docs (str): The input text to add to the tree.
        """
        # if self.tree is not None:
        #     user_input = input(
        #         "Warning: Overwriting existing tree. Did you mean to call 'add_to_existing' instead? (y/n): "
        #     )
        #     if user_input.lower() == "y":
        #         # self.add_to_existing(docs)
        #         return
        assert self.tree is None, "Tree already exists"

        self.tree = self.tree_builder.build_from_text(text=docs)
        self.retriever = TreeRetriever(
            self.tree_retriever_config,
            self.tree,
            retrieve_context_func=self.retrieve_context_func
        )

    def retrieve(
        self,
        question,
        start_layer: int = None,
        num_layers: int = None,
        top_k: int = 10,
        max_tokens: int = 3500,
        collapse_tree: bool = True,
        return_layer_information: bool = True,
    ):
        """
        Retrieves information and answers a question using the TreeRetriever instance.

        Args:
            question (str): The question to answer.
            start_layer (int): The layer to start from. Defaults to self.start_layer.
            num_layers (int): The number of layers to traverse. Defaults to self.num_layers.
            max_tokens (int): The maximum number of tokens. Defaults to 3500.
            use_all_information (bool): Whether to retrieve information from all nodes. Defaults to False.

        Returns:
            str: The context from which the answer can be found.

        Raises:
            ValueError: If the TreeRetriever instance has not been initialized.
        """
        if self.retriever is None:
            raise ValueError(
                "The TreeRetriever instance has not been initialized. Call 'add_documents' first."
            )

        return self.retriever.retrieve(
            question,
            start_layer,
            num_layers,
            top_k,
            max_tokens,
            collapse_tree,
            return_layer_information,
        )

    def answer_question(
        self,
        question,
        top_k: int = 1,
        start_layer: int = 1,
        num_layers: int = None,
        max_tokens: int = 3500,
        collapse_tree: bool = False,
        # return_layer_information: bool = False,
    ):
        """
        Retrieves information and answers a question using the TreeRetriever instance.

        Args:
            question (str): The question to answer.
            start_layer (int): The layer to start from. Defaults to self.start_layer.
            num_layers (int): The number of layers to traverse. Defaults to self.num_layers.
            max_tokens (int): The maximum number of tokens. Defaults to 3500.
            use_all_information (bool): Whether to retrieve information from all nodes. Defaults to False.
            ??

        Returns:
            str: The answer to the question.

        Raises:
            ValueError: If the TreeRetriever instance has not been initialized.
        """
        # if return_layer_information:
        # context, layer_information = self.retrieve(
        #     question, start_layer, num_layers, top_k, max_tokens, collapse_tree, True
        # )

        context = self.retrieve(
            question, start_layer, num_layers, top_k, max_tokens, collapse_tree, True
        )

        answer = self.qa_model.answer_question(context, question)

        # if return_layer_information:
        #     return answer, layer_information

        return answer

    def save(self, path):
        if self.tree is None:
            raise ValueError("There is no tree to save.")
        with open(path, "wb") as file:
            pickle.dump(self.tree, file)
        logging.info(f"Tree successfully saved to {path}")


# class TreeRetrieverConfig:
#     def __init__(
#         self,
#         tokenizer=None,
#         threshold=None,
#         top_k=None,
#         selection_mode=None,
#         context_embedding_model=None,
#         embedding_model=None,
#         num_layers=None,
#         start_layer=None,
#         retrieve_context_func=None,
#     ):
#         self.retrieve_context_func = retrieve_context_func
#         assert self.retrieve_context_func is not None, "retrieve_context_func must be provided"

#         if tokenizer is None:
#             tokenizer = tiktoken.get_encoding("cl100k_base")
#         self.tokenizer = tokenizer

#         if threshold is None:
#             threshold = 0.5
#         if not isinstance(threshold, float) or not (0 <= threshold <= 1):
#             raise ValueError("threshold must be a float between 0 and 1")
#         self.threshold = threshold

#         if top_k is None:
#             top_k = 5
#         if not isinstance(top_k, int) or top_k < 1:
#             raise ValueError("top_k must be an integer and at least 1")
#         self.top_k = top_k

#         if selection_mode is None:
#             selection_mode = "top_k"
#         if not isinstance(selection_mode, str) or selection_mode not in [
#             "top_k",
#             "threshold",
#         ]:
#             raise ValueError(
#                 "selection_mode must be a string and either 'top_k' or 'threshold'"
#             )
#         self.selection_mode = selection_mode

#         if context_embedding_model is None:
#             context_embedding_model = "OpenAI"
#         if not isinstance(context_embedding_model, str):
#             raise ValueError("context_embedding_model must be a string")
#         self.context_embedding_model = context_embedding_model

#         # if embedding_model is None:
#         #     embedding_model = OpenAIEmbeddingModel()
#         assert embedding_model is not None, "embedding_model must be provided"
#         if not isinstance(embedding_model, BaseEmbeddingModel):
#             raise ValueError(
#                 "embedding_model must be an instance of BaseEmbeddingModel"
#             )
#         self.embedding_model = embedding_model

#         if num_layers is not None:
#             if not isinstance(num_layers, int) or num_layers < 0:
#                 raise ValueError("num_layers must be an integer and at least 0")
#         self.num_layers = num_layers

#         if start_layer is not None:
#             if not isinstance(start_layer, int) or start_layer < 0:
#                 raise ValueError("start_layer must be an integer and at least 0")
#         self.start_layer = start_layer

#     def log_config(self):
#         config_log = """
#         TreeRetrieverConfig:
#             Tokenizer: {tokenizer}
#             Threshold: {threshold}
#             Top K: {top_k}
#             Selection Mode: {selection_mode}
#             Context Embedding Model: {context_embedding_model}
#             Embedding Model: {embedding_model}
#             Num Layers: {num_layers}
#             Start Layer: {start_layer}
#         """.format(
#             tokenizer=self.tokenizer,
#             threshold=self.threshold,
#             top_k=self.top_k,
#             selection_mode=self.selection_mode,
#             context_embedding_model=self.context_embedding_model,
#             embedding_model=self.embedding_model,
#             num_layers=self.num_layers,
#             start_layer=self.start_layer,
#         )
#         return config_log


class TreeRetriever(BaseRetriever):

    def __init__(self, config, tree, retrieve_context_func=None) -> None:
        if not isinstance(tree, Tree):
            raise ValueError("tree must be an instance of Tree")

        if config.num_layers is not None and config.num_layers > tree.num_layers + 1:
            raise ValueError(
                "num_layers in config must be less than or equal to tree.num_layers + 1"
            )

        if config.start_layer is not None and config.start_layer > tree.num_layers:
            raise ValueError(
                "start_layer in config must be less than or equal to tree.num_layers"
            )

        self.tree = tree
        self.num_layers = (
            config.num_layers if config.num_layers is not None else tree.num_layers + 1
        )
        self.start_layer = (
            config.start_layer if config.start_layer is not None else tree.num_layers
        )

        if self.num_layers > self.start_layer + 1:
            raise ValueError("num_layers must be less than or equal to start_layer + 1")

        self.tokenizer = config.tokenizer
        self.top_k = config.top_k
        self.threshold = config.threshold
        self.selection_mode = config.selection_mode
        self.embedding_model = config.embedding_model
        self.context_embedding_model = config.context_embedding_model

        self.tree_node_index_to_layer = reverse_mapping(self.tree.layer_to_nodes)

        logging.info(
            f"Successfully initialized TreeRetriever with Config {config.log_config()}"
        )
        self.retrieve_context_func = retrieve_context_func
        assert self.retrieve_context_func is not None, "retrieve_context_func must be provided"

    def create_embedding(self, text: str) -> List[float]:
        """
        Generates embeddings for the given text using the specified embedding model.

        Args:
            text (str): The text for which to generate embeddings.

        Returns:
            List[float]: The generated embeddings.
        """
        return self.embedding_model.create_embedding(text)

    # def retrieve_information_collapse_tree(self, query: str, top_k: int, max_tokens: int) -> str:
    #     """
    #     Retrieves the most relevant information from the tree based on the query.

    #     Args:
    #         query (str): The query text.
    #         max_tokens (int): The maximum number of tokens.

    #     Returns:
    #         str: The context created using the most relevant nodes.
    #     """

    #     query_embedding = self.create_embedding(query)

    #     selected_nodes = []

    #     node_list = get_node_list(self.tree.all_nodes)

    #     embeddings = get_embeddings(node_list, self.context_embedding_model)

    #     distances = distances_from_embeddings(query_embedding, embeddings)

    #     indices = indices_of_nearest_neighbors_from_distances(distances)

    #     total_tokens = 0
    #     for idx in indices[:top_k]:

    #         node = node_list[idx]
    #         node_tokens = len(self.tokenizer.encode(node.text))

    #         if total_tokens + node_tokens > max_tokens:
    #             break

    #         selected_nodes.append(node)
    #         total_tokens += node_tokens

    #     context = get_text(selected_nodes)
    #     return selected_nodes, context

    # def retrieve_information(
    #     self, current_nodes: List[Node], query: str, num_layers: int, top_k: int = None
    # ) -> str:
    #     """
    #     Retrieves the most relevant information from the tree based on the query.

    #     Args:
    #         current_nodes (List[Node]): A List of the current nodes.
    #         query (str): The query text.
    #         num_layers (int): The number of layers to traverse.

    #     Returns:
    #         str: The context created using the most relevant nodes.
    #     """

    #     query_embedding = self.create_embedding(query)

    #     selected_nodes = []

    #     node_list = current_nodes

    #     if top_k is None:
    #         top_k = self.top_k

    #     for layer in range(num_layers):

    #         embeddings = get_embeddings(node_list, self.context_embedding_model)

    #         distances = distances_from_embeddings(query_embedding, embeddings)

    #         indices = indices_of_nearest_neighbors_from_distances(distances)

    #         if self.selection_mode == "threshold":
    #             best_indices = [
    #                 index for index in indices if distances[index] > self.threshold
    #             ]

    #         elif self.selection_mode == "top_k":
    #             best_indices = indices[: top_k]

    #         nodes_to_add = [node_list[idx] for idx in best_indices]

    #         selected_nodes.extend(nodes_to_add)

    #         if layer != num_layers - 1:

    #             child_nodes = []

    #             for index in best_indices:
    #                 child_nodes.extend(node_list[index].children)

    #             # take the unique values
    #             child_nodes = list(dict.fromkeys(child_nodes))
    #             node_list = [self.tree.all_nodes[i] for i in child_nodes]

    #     context = get_text(selected_nodes)
    #     return selected_nodes, context

    def retrieve(
        self,
        query: str,
        start_layer: int = None,
        num_layers: int = None,
        top_k: int = 10,
        max_tokens: int = 3500,
        collapse_tree: bool = True,
        return_layer_information: bool = False,
    ) -> str:
        """
        Queries the tree and returns the most relevant information.

        Args:
            query (str): The query text.
            start_layer (int): The layer to start from. Defaults to self.start_layer.
            num_layers (int): The number of layers to traverse. Defaults to self.num_layers.
            max_tokens (int): The maximum number of tokens. Defaults to 3500.
            collapse_tree (bool): Whether to retrieve information from all nodes. Defaults to False.

        Returns:
            str: The result of the query.
        """

        if not isinstance(query, str):
            raise ValueError("query must be a string")

        if not isinstance(max_tokens, int) or max_tokens < 1:
            raise ValueError("max_tokens must be an integer and at least 1")

        if not isinstance(collapse_tree, bool):
            raise ValueError("collapse_tree must be a boolean")

        # Set defaults
        start_layer = self.start_layer if start_layer is None else start_layer
        num_layers = self.num_layers if num_layers is None else num_layers

        if not isinstance(start_layer, int) or not (
            0 <= start_layer <= self.tree.num_layers
        ):
            raise ValueError(
                "start_layer must be an integer between 0 and tree.num_layers"
            )

        if not isinstance(num_layers, int) or num_layers < 1:
            raise ValueError("num_layers must be an integer and at least 1")

        if num_layers > (start_layer + 1):
            raise ValueError("num_layers must be less than or equal to start_layer + 1")

        # if collapse_tree:
        #     logging.info(f"Using collapsed_tree")
        #     selected_nodes, context = self.retrieve_information_collapse_tree(
        #         query, top_k, max_tokens
        #     )
        # else:
        #     layer_nodes = self.tree.layer_to_nodes[start_layer]
        #     selected_nodes, context = self.retrieve_information(
        #         layer_nodes,
        #         query,
        #         num_layers,
        #         top_k=top_k
        #     )

        # if return_layer_information:

        #     layer_information = []

        #     for node in selected_nodes:
        #         layer_information.append(
        #             {
        #                 "node_index": node.index,
        #                 "layer_number": self.tree_node_index_to_layer[node.index],
        #             }
        #         )

        #     return context, layer_information

        context = self.retrieve_context_func(query, self.tree, self.embedding_model)

        return context


class TreeBuilder:
    """
    The TreeBuilder class is responsible for building a hierarchical text abstraction
    structure, known as a "tree," using summarization models and
    embedding models.
    """

    def __init__(self, config) -> None:
        """Initializes the tokenizer, maximum tokens, number of layers, top-k value, threshold, and selection mode."""

        self.tokenizer = config.tokenizer
        self.max_tokens = config.max_tokens
        self.num_layers = config.num_layers
        self.top_k = config.top_k
        self.threshold = config.threshold
        self.selection_mode = config.selection_mode
        self.summarization_length = config.summarization_length
        self.summarization_model = config.summarization_model
        self.embedding_models = config.embedding_models
        self.cluster_embedding_model = config.cluster_embedding_model

        # logging.info(
        #     f"Successfully initialized TreeBuilder with Config {config.log_config()}"
        # )

        if not isinstance(config, ClusterTreeConfig):
            raise ValueError("config must be an instance of ClusterTreeConfig")
        self.reduction_dimension = config.reduction_dimension
        self.clustering_algorithm = config.clustering_algorithm
        self.clustering_params = config.clustering_params

        logging.info(
            f"Successfully initialized TreeBuilder with Config {config.log_config()}"
        )

    def create_node(
        self, index: int, text: str, children_indices: Optional[Set[int]] = None, summarization_prompt: Optional[str] = None
    ) -> Tuple[int, Node]:
        """Creates a new node with the given index, text, and (optionally) children indices.

        Args:
            index (int): The index of the new node.
            text (str): The text associated with the new node.
            children_indices (Optional[Set[int]]): A set of indices representing the children of the new node.
                If not provided, an empty set will be used.

        Returns:
            Tuple[int, Node]: A tuple containing the index and the newly created node.
        """
        if children_indices is None:
            children_indices = set()

        embeddings = {
            model_name: model.create_embedding(text)
            for model_name, model in self.embedding_models.items()
        }
        return (index, Node(text, index, children_indices, embeddings, summarization_prompt))

    def create_embedding(self, text) -> List[float]:
        """
        Generates embeddings for the given text using the specified embedding model.

        Args:
            text (str): The text for which to generate embeddings.

        Returns:
            List[float]: The generated embeddings.
        """
        return self.embedding_models[self.cluster_embedding_model].create_embedding(
            text
        )

    def summarize(self, context, max_tokens=150) -> str:
        """
        Generates a summary of the input context using the specified summarization model.

        Args:
            context (str, optional): The context to summarize.
            max_tokens (int, optional): The maximum number of tokens in the generated summary. Defaults to 150.o

        Returns:
            str: The generated summary.
        """
        return self.summarization_model.summarize(context, max_tokens)

    # def get_relevant_nodes(self, current_node, list_nodes) -> List[Node]:
    #     """
    #     Retrieves the top-k most relevant nodes to the current node from the list of nodes
    #     based on cosine distance in the embedding space.

    #     Args:
    #         current_node (Node): The current node.
    #         list_nodes (List[Node]): The list of nodes.

    #     Returns:
    #         List[Node]: The top-k most relevant nodes.
    #     """
    #     embeddings = get_embeddings(
    #         list_nodes,
    #         embedding_model_name="EMB"
    #     )
    #     distances = distances_from_embeddings(
    #         current_node.embeddings[self.cluster_embedding_model], embeddings
    #     )
    #     indices = indices_of_nearest_neighbors_from_distances(distances)

    #     if self.selection_mode == "threshold":
    #         best_indices = [
    #             index for index in indices if distances[index] > self.threshold
    #         ]

    #     elif self.selection_mode == "top_k":
    #         best_indices = indices[: self.top_k]

    #     nodes_to_add = [list_nodes[idx] for idx in best_indices]

    #     return nodes_to_add

    # def multithreaded_create_leaf_nodes(self, chunks: List[str]) -> Dict[int, Node]:
    #     """Creates leaf nodes using multithreading from the given list of text chunks.

    #     Args:
    #         chunks (List[str]): A list of text chunks to be turned into leaf nodes.

    #     Returns:
    #         Dict[int, Node]: A dictionary mapping node indices to the corresponding leaf nodes.
    #     """
    #     with ThreadPoolExecutor() as executor:
    #         future_nodes = {
    #             executor.submit(self.create_node, index, text): (index, text)
    #             for index, text in enumerate(chunks)
    #         }

    #         leaf_nodes = {}
    #         for future in as_completed(future_nodes):
    #             index, node = future.result()
    #             leaf_nodes[index] = node

    #     return leaf_nodes

    def build_from_text(self, text: str) -> Tree:
        """Builds a golden tree from the input text, optionally using multithreading.

        Args:
            text (str): The input text.

        Returns:
            Tree: The golden tree structure.
        """
        chunks = split_text(text, self.tokenizer, self.max_tokens)

        logging.info("Creating Leaf Nodes")

        # if use_multithreading:
        #     leaf_nodes = self.multithreaded_create_leaf_nodes(chunks)
        # else:
        leaf_nodes = {}
        for index, text in enumerate(chunks):
            __, node = self.create_node(index, text)
            leaf_nodes[index] = node

        layer_to_nodes = {0: list(leaf_nodes.values())}

        logging.info(f"Created {len(leaf_nodes)} Leaf Embeddings")

        logging.info("Building All Nodes")

        all_nodes = copy.deepcopy(leaf_nodes)

        root_nodes = self.construct_tree(all_nodes, all_nodes, layer_to_nodes)

        tree = Tree(all_nodes, root_nodes, leaf_nodes, self.num_layers, layer_to_nodes)

        return tree

    # @abstractclassmethod
    # def construct_tree(
    #     self,
    #     current_level_nodes: Dict[int, Node],
    #     all_tree_nodes: Dict[int, Node],
    #     layer_to_nodes: Dict[int, List[Node]],
    #     use_multithreading: bool = True,
    # ) -> Dict[int, Node]:
    #     """
    #     Constructs the hierarchical tree structure layer by layer by iteratively summarizing groups
    #     of relevant nodes and updating the current_level_nodes and all_tree_nodes dictionaries at each step.

    #     Args:
    #         current_level_nodes (Dict[int, Node]): The current set of nodes.
    #         all_tree_nodes (Dict[int, Node]): The dictionary of all nodes.
    #         use_multithreading (bool): Whether to use multithreading to speed up the process.

    #     Returns:
    #         Dict[int, Node]: The final set of root nodes.
    #     """
    #     pass

    #     # logging.info("Using Transformer-like TreeBuilder")

    #     # def process_node(idx, current_level_nodes, new_level_nodes, all_tree_nodes, next_node_index, lock):
    #     #     relevant_nodes_chunk = self.get_relevant_nodes(
    #     #         current_level_nodes[idx], current_level_nodes
    #     #     )

    #     #     node_texts = get_text(relevant_nodes_chunk)

    #     #     summarized_text = self.summarize(
    #     #         context=node_texts,
    #     #         max_tokens=self.summarization_length,
    #     #     )

    #     #     logging.info(
    #     #         f"Node Texts Length: {len(self.tokenizer.encode(node_texts))}, Summarized Text Length: {len(self.tokenizer.encode(summarized_text))}"
    #     #     )

    #     #     next_node_index, new_parent_node = self.create_node(
    #     #         next_node_index,
    #     #         summarized_text,
    #     #         {node.index for node in relevant_nodes_chunk}
    #     #     )

    #     #     with lock:
    #     #         new_level_nodes[next_node_index] = new_parent_node

    #     # for layer in range(self.num_layers):
    #     #     logging.info(f"Constructing Layer {layer}: ")

    #     #     node_list_current_layer = get_node_list(current_level_nodes)
    #     #     next_node_index = len(all_tree_nodes)

    #     #     new_level_nodes = {}
    #     #     lock = Lock()

    #     #     if use_multithreading:
    #     #         with ThreadPoolExecutor() as executor:
    #     #             for idx in range(0, len(node_list_current_layer)):
    #     #                 executor.submit(process_node, idx, node_list_current_layer, new_level_nodes, all_tree_nodes, next_node_index, lock)
    #     #                 next_node_index += 1
    #     #             executor.shutdown(wait=True)
    #     #     else:
    #     #         for idx in range(0, len(node_list_current_layer)):
    #     #             process_node(idx, node_list_current_layer, new_level_nodes, all_tree_nodes, next_node_index, lock)

    #     #     layer_to_nodes[layer + 1] = list(new_level_nodes.values())
    #     #     current_level_nodes = new_level_nodes
    #     #     all_tree_nodes.update(new_level_nodes)

    #     # return new_level_nodes

    def construct_tree(
        self,
        current_level_nodes: Dict[int, Node],
        all_tree_nodes: Dict[int, Node],
        layer_to_nodes: Dict[int, List[Node]],
        # use_multithreading: bool = False,
    ) -> Dict[int, Node]:
        logging.info("Using Cluster TreeBuilder")

        next_node_index = len(all_tree_nodes)

        def process_cluster(
            cluster,
            new_level_nodes,
            next_node_index,
            summarization_length,
            # lock
        ):
            node_texts = get_text(cluster)

            summarized_text = self.summarize(
                context=node_texts,
                max_tokens=summarization_length,
            )

            if isinstance(summarized_text, tuple):
                assert len(summarized_text) == 2
                summarized_text, summarization_prompt = summarized_text
            else:
                summarization_prompt = None

            logging.info(
                f"Node Texts Length: {len(self.tokenizer.encode(node_texts))}, Summarized Text Length: {len(self.tokenizer.encode(summarized_text))}"
            )

            __, new_parent_node = self.create_node(
                index=next_node_index,
                text=summarized_text,
                children_indices={node.index for node in cluster},
                summarization_prompt=summarization_prompt
            )

            # with lock:
            new_level_nodes[next_node_index] = new_parent_node

        for layer in range(self.num_layers):

            new_level_nodes = {}

            logging.info(f"Constructing Layer {layer}")

            node_list_current_layer = get_node_list(current_level_nodes)

            if len(node_list_current_layer) <= self.reduction_dimension + 1:
                self.num_layers = layer
                logging.info(
                    f"Stopping Layer construction: Cannot Create More Layers. Total Layers in tree: {layer}"
                )
                break

            clusters = self.clustering_algorithm.perform_clustering(
                node_list_current_layer,
                self.cluster_embedding_model,
                reduction_dimension=self.reduction_dimension,
                **self.clustering_params,
            )

            # lock = Lock()

            summarization_length = self.summarization_length
            logging.info(f"Summarization Length: {summarization_length}")

            # if use_multithreading:
            #     with ThreadPoolExecutor() as executor:
            #         for cluster in clusters:
            #             executor.submit(
            #                 process_cluster,
            #                 cluster,
            #                 new_level_nodes,
            #                 next_node_index,
            #                 summarization_length,
            #                 lock,
            #             )
            #             next_node_index += 1
            #         executor.shutdown(wait=True)

            # else:
            for cluster in clusters:
                process_cluster(
                    cluster,
                    new_level_nodes,
                    next_node_index,
                    summarization_length,
                    # lock,
                )
                next_node_index += 1

            layer_to_nodes[layer + 1] = list(new_level_nodes.values())
            current_level_nodes = new_level_nodes
            all_tree_nodes.update(new_level_nodes)

            tree = Tree(
                all_tree_nodes,
                layer_to_nodes[layer + 1],
                layer_to_nodes[0],
                layer + 1,
                layer_to_nodes,
            )

        return current_level_nodes


# class ClusterTreeBuilder(TreeBuilder):
#     def __init__(self, config) -> None:
#         super().__init__(config)

#         if not isinstance(config, ClusterTreeConfig):
#             raise ValueError("config must be an instance of ClusterTreeConfig")
#         self.reduction_dimension = config.reduction_dimension
#         self.clustering_algorithm = config.clustering_algorithm
#         self.clustering_params = config.clustering_params

#         logging.info(
#             f"Successfully initialized ClusterTreeBuilder with Config {config.log_config()}"
#         )

    # def construct_tree(
    #     self,
    #     current_level_nodes: Dict[int, Node],
    #     all_tree_nodes: Dict[int, Node],
    #     layer_to_nodes: Dict[int, List[Node]],
    #     # use_multithreading: bool = False,
    # ) -> Dict[int, Node]:
    #     logging.info("Using Cluster TreeBuilder")

    #     next_node_index = len(all_tree_nodes)

    #     def process_cluster(
    #         cluster,
    #         new_level_nodes,
    #         next_node_index,
    #         summarization_length,
    #         # lock
    #     ):
    #         node_texts = get_text(cluster)

    #         summarized_text = self.summarize(
    #             context=node_texts,
    #             max_tokens=summarization_length,
    #         )

    #         if isinstance(summarized_text, tuple):
    #             assert len(summarized_text) == 2
    #             summarized_text, summarization_prompt = summarized_text
    #         else:
    #             summarization_prompt = None

    #         logging.info(
    #             f"Node Texts Length: {len(self.tokenizer.encode(node_texts))}, Summarized Text Length: {len(self.tokenizer.encode(summarized_text))}"
    #         )

    #         __, new_parent_node = self.create_node(
    #             index=next_node_index,
    #             text=summarized_text,
    #             children_indices={node.index for node in cluster},
    #             summarization_prompt=summarization_prompt
    #         )

    #         # with lock:
    #         new_level_nodes[next_node_index] = new_parent_node

    #     for layer in range(self.num_layers):

    #         new_level_nodes = {}

    #         logging.info(f"Constructing Layer {layer}")

    #         node_list_current_layer = get_node_list(current_level_nodes)

    #         if len(node_list_current_layer) <= self.reduction_dimension + 1:
    #             self.num_layers = layer
    #             logging.info(
    #                 f"Stopping Layer construction: Cannot Create More Layers. Total Layers in tree: {layer}"
    #             )
    #             break

    #         clusters = self.clustering_algorithm.perform_clustering(
    #             node_list_current_layer,
    #             self.cluster_embedding_model,
    #             reduction_dimension=self.reduction_dimension,
    #             **self.clustering_params,
    #         )

    #         # lock = Lock()

    #         summarization_length = self.summarization_length
    #         logging.info(f"Summarization Length: {summarization_length}")

    #         # if use_multithreading:
    #         #     with ThreadPoolExecutor() as executor:
    #         #         for cluster in clusters:
    #         #             executor.submit(
    #         #                 process_cluster,
    #         #                 cluster,
    #         #                 new_level_nodes,
    #         #                 next_node_index,
    #         #                 summarization_length,
    #         #                 lock,
    #         #             )
    #         #             next_node_index += 1
    #         #         executor.shutdown(wait=True)

    #         # else:
    #         for cluster in clusters:
    #             process_cluster(
    #                 cluster,
    #                 new_level_nodes,
    #                 next_node_index,
    #                 summarization_length,
    #                 # lock,
    #             )
    #             next_node_index += 1

    #         layer_to_nodes[layer + 1] = list(new_level_nodes.values())
    #         current_level_nodes = new_level_nodes
    #         all_tree_nodes.update(new_level_nodes)

    #         tree = Tree(
    #             all_tree_nodes,
    #             layer_to_nodes[layer + 1],
    #             layer_to_nodes[0],
    #             layer + 1,
    #             layer_to_nodes,
    #         )

    #     return current_level_nodes

# Setup

In [7]:
# Cinderella story defined in sample.txt
sample_text_path = os.path.join(SAMPLE_TEXT_PATH)
with open(sample_text_path, 'r') as file:
    text = file.read()

login(token=HF_TOKEN)

# tree_path = TREE_PATH
rac = RetrievalAugmentationConfig(
    summarization_model=GEMMASummarizationModel(),
    qa_model=GEMMAQAModel(),
    embedding_model=SBertEmbeddingModel(),
    tb_max_tokens=GRANULARITY,
    tb_summarization_length=SUMMARY_LENGTH
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /home/oh/arubinstein17/.cache/huggingface/token
Login successful


/home/oh/arubinstein17/github/raptor/envs/raptor/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.27s/it]
2025-02-27 01:45:33,785 - Load pretrained SentenceTransformer: sentence-transformers/multi-qa-mpnet-base-cos-v1
2025-02-27 01:45:35,705 - Use pytorch device: cuda


# Standard RAG

### Create flat-database

In [ ]:
# create flat-database
# retrieve from flat-database

In [ ]:
ra = RetrievalAugmentation(
    config=rac,
    tree=TREE_PATH,
    retrieve_context_func=retrieve_information_from_flat_database??
    create_database_func=create_flat_database??
)

In [ ]:
ra.add_documents(text)

### Answer question

In [ ]:
question = "What was the cause of Evelyn's symptoms?"

answer = ra.answer_question(
    question=question,
    # top_k=1,
    # collapse_tree=False,
    # start_layer=1, # goes from top to bottom (descending); from this layer the seeing set of nodes is taken
)

print("Answer: ", answer)

# RAPTOR-based RAG

### [Task] Retrieval from tree

In [10]:
def retrieve_information_from_tree(
        query: str, database: Any, embedding_func: Callable
    ) -> str:
        """
        Retrieves the most relevant information from the tree based on the query.

        Args:
            query (str): The query text.
            database (Any): The database with documents.
            embedding_func (Callable): The function to embed the documents.

        Returns:
            str: The context created using the most relevant documents.
        """

        # INSERT YOUR CODE BELOW
        print("SOLUTION FOR retrieve_information")
        start_layer = 1
        num_layers = 2
        top_k = 1

        node_list = database.layer_to_nodes[start_layer]

        # query_embedding = self.create_embedding(query)
        query_embedding = embedding_func(query)

        selected_nodes = []

        # node_list = current_nodes

        # if top_k is None:
        #     top_k = self.top_k

        for layer in range(num_layers):

            # embeddings = get_embeddings(node_list, self.context_embedding_model)
            embeddings = get_embeddings(
                node_list,
                embedding_model_name="EMB"
            )

            distances = distances_from_embeddings(query_embedding, embeddings)

            indices = indices_of_nearest_neighbors_from_distances(distances)

            # if self.selection_mode == "threshold":
            #     best_indices = [
            #         index for index in indices if distances[index] > self.threshold
            #     ]

            # elif self.selection_mode == "top_k":
            best_indices = indices[: top_k]

            nodes_to_add = [node_list[idx] for idx in best_indices]

            selected_nodes.extend(nodes_to_add)

            if layer != num_layers - 1:

                child_nodes = []

                for index in best_indices:
                    child_nodes.extend(node_list[index].children)

                # take the unique values
                child_nodes = list(dict.fromkeys(child_nodes))
                node_list = [database.all_nodes[i] for i in child_nodes]

        context = get_text(selected_nodes)
        # INSERT YOUR CODE ABOVE
        # return selected_nodes, context
        return context


def get_embeddings(node_list: List[Node], embedding_model_name: str) -> List:
    """
    Extracts the embeddings of nodes from a list of nodes.

    Args:
        node_list (List[Node]): List of nodes to extract embeddings from.
        embedding_model_name (str): Name of embedding model to use.

    Returns:
        List: List of node embeddings.
    """
    return [node.embeddings[embedding_model_name] for node in node_list]

### Create tree

In [11]:
ra = RetrievalAugmentation(
    config=rac,
    tree=TREE_PATH,
    retrieve_context_func=retrieve_information_from_tree
)

2025-02-27 01:47:07,808 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 200
            Summarization Model: <__main__.GEMMASummarizationModel object at 0x7ff707e0bc10>
            Embedding Models: {'EMB': <__main__.SBertEmbeddingModel object at 0x7ff716075d80>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2025-02-27 01:47:07,808 - Successfully initialized RetrievalAugmentation with Config 
        RetrievalAugmentationConfig:
            
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
 

In [12]:
ra.add_documents(text)

2025-02-27 01:47:11,971 - Creating Leaf Nodes
Batches: 100%|██████████| 1/1 [00:00<00:00, 118.57it/s]
2025-02-27 01:47:14,402 - Created 19 Leaf Embeddings
2025-02-27 01:47:14,402 - Building All Nodes
2025-02-27 01:47:14,404 - Using Cluster TreeBuilder
2025-02-27 01:47:14,404 - Constructing Layer 0
/home/oh/arubinstein17/github/raptor/envs/raptor/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/oh/arubinstein17/github/raptor/envs/raptor/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
2025-02-27 01:47:24,956 - Summarization Length: 200
/home/oh/arubinstein17/github/raptor/envs/raptor/lib/python3.10/site-packages/transformers/generation/configura

### Save tree

In [ ]:
tree_path = TREE_PATH

rac = RetrievalAugmentationConfig(
    summarization_model=GEMMASummarizationModel(),
    qa_model=GEMMAQAModel(),
    embedding_model=SBertEmbeddingModel(),
    tr_threshold=0.9,
    tb_max_tokens=100,
    tb_summarization_length=200
)

ra = RetrievalAugmentation(config=rac, tree=tree_path)

if not os.path.exists(tree_path):
    # construct the tree
    ra.add_documents(text)

    ra.save(tree_path)

### Answer question

In [13]:
question = "What was the cause of Evelyn's symptoms?"

answer = ra.answer_question(
    question=question,
    # top_k=1,
    # collapse_tree=False,
    # start_layer=1, # goes from top to bottom (descending); from this layer the seeing set of nodes is taken
)

print("Answer: ", answer)

SOLUTION FOR retrieve_information


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.91it/s]

Context:  Sure, here's a summary of the passage:  Evelyn, a seemingly healthy individual, began exhibiting peculiar symptoms that gradually progressed from heightened senses to more pronounced behavioral transformations. Her symptoms included an uncanny sense of direction, increased visual acuity, and a compulsive need to coo softly, particularly during twilight hours.  The symptoms were initially observed by neighbors, but they were initially dismissed as harmless. However, as the symptoms persisted, they became more pronounced, leading to identity struggles and social isolation.  A collaboration between Dr. Clara Novak and Dr. Liam Greer revealed that the cause of Evelyn's symptoms was a fungus called Columba benedicta, which released spores that affected the human nervous system.  Treatment involved antifungal medications, cognitive-behavioral therapy, nasal spray, and topical cream. Additional treatments included intravenous antifungal therapy for severe cases.  The condition was m

Answer:  The cause of Evelyn's symptoms was a fungus called Columba benedicta, which released spores that affected the human nervous system.
